# Hash Generation
Reads every **active** row in `hash_config`, hashes each configured source table, and writes the results to the `hashed_values` output table.

For every configured table:
- **primary_key_hash_value** = hash of the primary key column's normalized value
- **composite_keys_hash_value** = hash of the concatenated (no delimiter) normalized composite key column values
- **hash_value** = hash of the concatenated (no delimiter) normalized values of *every* column in the table, as reported by `information_schema.columns`, in column order, **excluding any column whose name ends in `___c`**
- the hash algorithm (`md5` or `sha256`) is whatever `hash_algorithm` says for that table

### Column normalization rules (per `Hash_Generation_Logic` spec)
Before concatenation, each column value is converted/normalized based on its datatype, mirroring the source `ROW_MD5` procedure:

| Data type | Normalization |
|---|---|
| `date` | Formatted as `YYYY-MM-DD HH24:MI:SS` |
| `timestamp` / `timestamp_ntz` | Formatted as `YYYY-MM-DD HH24:MI:SS` |
| `string` / everything else | Cast to text; long values truncated to the first 32,767 characters (CLOB rule); `NULL` → blank; `CHR(13)`/`CHR(10)` (CR/LF) removed; embedded `"` doubled to `""`; `CHR(0)` removed |

Values are concatenated with **no delimiter**, in database column order (`ordinal_position`), then hashed and converted to lowercase hex (Spark's `md5`/`sha2` already return lowercase hex, matching `DBMS_CRYPTO.HASH`).


## Imports

In [0]:
%run ./00_config

## Read active config rows

In [0]:
config_rows = (
    spark.table(config_table)
    .filter(F.col("is_active") == 1)
    .collect()
)

print(f"Found {len(config_rows)} active table(s) to process")

## Helper functions
Normalizes each column's value according to its datatype before concatenating, per the `Hash_Generation_Logic` rules:
- `date` → `YYYY-MM-DD HH24:MI:SS`; `timestamp` → `YYYY-MM-DD HH24:MI:SS.FF`
- everything else → cast to text, truncated to 32,767 chars (CLOB rule), `NULL` → blank (`''`), CR/LF removed, `"` escaped to `""`, `CHR(0)` removed

Columns are then concatenated with no delimiter


In [0]:
def hash_expr(algorithm, expr):
    """Return a hash column expression for the given algorithm."""
    algorithm = algorithm.strip().lower()
    if algorithm == "md5":
        return F.md5(expr)
    elif algorithm in ("sha256", "sha-256"):
        return F.sha2(expr, 256)
    else:
        raise ValueError(f"Unsupported hash_algorithm: {algorithm}")


def get_hashable_columns(catalog, schema, table):
    table = table.lower()
    src_schema=schema.lower()
    catalog=catalog.lower()
    print(f"Inside get_hashable_columns function-> Processing {catalog}.{schema}.{table}")
    cols_df = spark.sql(f"""
        SELECT column_name, data_type
        FROM {catalog}.information_schema.columns
        WHERE table_schema = '{schema}'
          AND table_name   = '{table}'
          AND column_name NOT RLIKE '___c$'
        ORDER BY ordinal_position
    """)
    l = [(r["column_name"], r["data_type"]) for r in cols_df.collect()]
    print("columns and dtypes:",l)
    return [(r["column_name"], r["data_type"]) for r in cols_df.collect()]


# DBMS_LOB.SUBSTR(value, 32767, 1) equivalent used for CLOB-like values
CLOB_CHAR_LIMIT = 32767


def standardize_column(colname, data_type):
    """
      DATE                     -> 'YYYY-MM-DD HH24:MI:SS'
      TIMESTAMP                -> 'YYYY-MM-DD HH24:MI:SS.FF'
      VARCHAR2 / CHAR / CLOB /
      any other datatype       -> TO_CHAR(value), then:
            - truncate to the first 32,767 characters (CLOB rule)
            - NULL           -> blank ('')
            - CHR(13) (CR)   -> removed
            - CHR(10) (LF)   -> removed
            - "              -> ""   (escaped, doubled)
            - CHR(0)         -> removed
    """
    dtype = (data_type or "").strip().lower()
    col = F.col(colname)

    # if dtype == "date":
    #     return F.coalesce(F.date_format(col, "yyyy-MM-dd HH:mm:ss"), F.lit(""))

    # if dtype.startswith("timestamp"):
    #     return F.coalesce(F.date_format(col, "yyyy-MM-dd HH:mm:ss.SSSSSS"), F.lit(""))

    if dtype == "date" or dtype.startswith("timestamp"):
        return F.coalesce(F.date_format(col, "yyyy-MM-dd HH:mm:ss"), F.lit(""))
        # return F.coalesce(F.date_format(col, "yyyy-MM-dd HH:mm:ss.SSSSSS"), F.lit(""))

    # VARCHAR2 / CHAR / CLOB / everything else -> TO_CHAR equivalent
    str_col = col.cast("string")

    # CLOB-style truncation (no-op for values shorter than the limit)
    str_col = F.when(
        F.length(str_col) > CLOB_CHAR_LIMIT,
        F.substring(str_col, 1, CLOB_CHAR_LIMIT)
    ).otherwise(str_col)

    # NULL -> blank
    str_col = F.coalesce(str_col, F.lit(""))

    # Remove CR / LF characters
    str_col = F.regexp_replace(str_col, "\r", "")
    str_col = F.regexp_replace(str_col, "\n", "")

    # Escape embedded double quotes: " -> ""
    str_col = F.regexp_replace(str_col, '"', '""')

    # Remove CHR(0) / NUL characters
    str_col = F.regexp_replace(str_col, "\u0000", "")

    return str_col


def concat_columns(col_specs):
    return F.concat(
        *[standardize_column(name, dtype) for name, dtype in col_specs]
    )


## Process each configured table

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

for row in config_rows:
    src_catalog    = row["catalog"]
    src_schema     = row["schema"]
    src_table      = row["table"]
    primary_key    = row["primary_key"].strip()
    composite_keys = [c.strip() for c in row["composite_keys"].split(",")]
    hash_algorithm = row["hash_algorithm"].strip().lower()

    full_table_name = f"{src_catalog}.{src_schema}.{src_table}"
    print(f"Processing {full_table_name} using {hash_algorithm}...")

    all_columns = get_hashable_columns(src_catalog, src_schema, src_table)
    if not all_columns:
        print(f"  WARNING: no hashable columns found for {full_table_name}, skipping")
        continue

    col_types = dict(all_columns)
    src_df = spark.table(full_table_name)

    pk_value_expr = concat_columns([(primary_key, col_types.get(primary_key, "STRING"))])
    ck_value_expr = concat_columns([(c, col_types.get(c, "STRING")) for c in composite_keys])
    row_value_expr = concat_columns(all_columns)

    result_df = src_df.select(
        F.lit(src_catalog).alias("catalog"),
        F.lit(src_schema).alias("schema"),
        F.lit(src_table).alias("table"),
        pk_value_expr.alias("primary_key"),
        hash_expr(hash_algorithm, pk_value_expr).alias("primary_key_hash_value"),
        ck_value_expr.alias("composite_keys"),
        hash_expr(hash_algorithm, ck_value_expr).alias("composite_keys_hash_value"),
        hash_expr(hash_algorithm, row_value_expr).alias("hash_value"),
        F.lit(hash_algorithm).alias("hash_algo"),
        F.current_timestamp().alias("loaded_at___c"),
    )

    # Temp - Refresh (rather than duplicate) this table rows in the output table
    spark.sql(f"""
        DELETE FROM {output_table}
        WHERE catalog = '{src_catalog}'
          AND schema  = '{src_schema}'
          AND table   = '{src_table}'
    """)

    (
        result_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(output_table)
    )

    row_count = result_df.count()
    print(f"  Appended {row_count} row(s) from {full_table_name} to {output_table}\n")

print("All configured tables processed.")

## Sanity check

In [0]:
display(spark.table(f"{tgt_catalog}.{tgt_schema}.{tgt_table}").limit(50))

In [0]:
%sql
-- SELECT * FROM iics.casemix.t_od_cs_tol_subvention_test

In [0]:
# df = spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/iics/casemix/history/hash_test/t_od_cs_training_subvention_test.csv")

In [0]:
# df.write.format("delta").mode("overwrite").saveAsTable("iics.casemix.t_od_cs_training_subvention_test")

In [0]:
# df.printSchema()